In [51]:
development_flag : bool = True

In [52]:
#import modules
import pandas as pd
import os

READ CSV FILE


In [ ]:
#create dataframe
dataframe : pd.DataFrame = pd.read_csv("archive/songs.csv")

SIMPLE DATA PREPROCESSING (remove newline chars, drop duplicates, remove short lyrics)


In [ ]:
def  lyricPreprocess (frame : pd.DataFrame )  -> pd.DataFrame:
    # Remove newline chars
    frame['lyrics'] = frame['lyrics'].str.replace(r"\\n", " ", regex=True)
    frame['lyrics'] = frame['lyrics'].str.replace("\n", " ", regex=True)
    frame['lyrics'] = frame['lyrics'].str.replace("\r", " ", regex=True)
    #convert everything to lowercase
    frame['lyrics'] = frame['lyrics'].str.lower()
    #drop duplicates 
    frame['lyrics'] = frame['lyrics'].drop_duplicates()
    #remove short lyrics
    frame : pd.DataFrame = frame[frame['lyrics'].notna() & (frame['lyrics'].str.len() > 250)]     
    
    return frame[['lyrics','genre']]

SEE IF DATASET IS BALANCED (ONLY FOR DEVELOPMENT)


In [ ]:
def getStatisticsOnDataset (frame : pd.DataFrame): 
    #count duplicates based on lyrics 
    counts : pd.Series[int] = frame['lyrics'].value_counts()
    counts : pd.Series[int ]= counts[counts > 1]
    print('Duplicates :' ,frame['lyrics'].duplicated().sum())
    genre_percent : pd.Series[float] = frame['genre'].value_counts(normalize=True) * 100
    print(genre_percent)

SUBSET OF GENRES


In [56]:
selectedGenres : list[str] = ['Rock', 'Pop', 'Electronic', 'Folk']
selectGenres2  : list[str] = [ 'Electronic','Hip-Hop','Jazz','Rock']
selectGenres3  : list[str] = ['Rock', 'Jazz', 'R&B','Hip-Hop']
selectGenres4  : list[str] = ['Rock', 'Jazz', 'Classical', 'Hip-Hop']
selectGenres5  : list[str] = ['Classical', 'Country','Electronic', 'Hip-Hop']

In [57]:
#choose the 2 subsets which the experiment is built on, one with higher f1-score,
# the other one with lower f1-score to see the model does not perform as good when the lyrics are semantically similar

sets : list[list[str]] = [selectGenres5, selectGenres2]

CREATE A BALANCED SUBSET


In [ ]:
def getBalancedSubset(frame : pd.DataFrame, subset :list[str])  -> pd.DataFrame:
    return frame[frame['genre'].isin(subset)].groupby('genre').sample(n=2500, random_state=69).reset_index(drop=True)

SAVE TO DISK (THE MODEL WILL USE THE SAME DATASETS, WHICH ARE DISTRIBUTED VIA A PUBLIC LINK SINCE GOOGLE COLAB CANNOT ACCESS THE DISK DIRECTLY WITHOUT FURTHER AUTHENTICATION)

In [59]:

def saveToDisk(frame : pd.DataFrame, subset : list[str]) :
    prefix : str = "balancedDatasets"
    folderPath : str = f'{prefix}/{"".join(subset)}'
    try:
        os.makedirs(folderPath)
    except FileExistsError : 
        pass
    
    path : str = f'{prefix}/{"".join(subset)}/balancedSubset10k{"".join(subset)}.csv'
    frame.to_csv(path, index=True, encoding='utf-8')
     

PIPELINE

1. PREPROCESS LYRICS
2. CREATE BALANCED SUBSET
3. PRINT STATISTICS (OPTIONAL)
4. SAVE DATASET TO CSV FOR FURTHER TASK

In [ ]:
def preProcessPipeLine(frame : pd.DataFrame, sets : list): 
    #preprocess the whole dataset
    frame : pd.DataFrame = lyricPreprocess(frame=frame)
    #iterate over both subsets needed for training
    for set in sets:
        subsetFrame : pd.DataFrame = frame
        #get a balanced dataset
        subsetFrame : pd.DataFrame = getBalancedSubset(frame=subsetFrame, subset=set)
        #print statistics (optional)
        if development_flag : 
            getStatisticsOnDataset(frame=subsetFrame)
        #save dataset to disk (maybe leave this out?)
        saveToDisk(frame=subsetFrame, subset=set)

RUN THE PIPELINE

In [61]:
preProcessPipeLine(frame=dataframe,sets=sets)

Duplicates : 0
genre
Classical     25.0
Country       25.0
Electronic    25.0
Hip-Hop       25.0
Name: proportion, dtype: float64
Duplicates : 0
genre
Electronic    25.0
Hip-Hop       25.0
Jazz          25.0
Rock          25.0
Name: proportion, dtype: float64
